# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franciskendrick/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in the modeling dataset represents a single daily performance snapshot for a specific content asset (`content_hash_id`) associated with a given client (`client_hash_id`) on a specific calendar date (`report_date`).

The historical evaluation slice is restricted to the mid-panel window of March 1, 2026 through March 31, 2026 (`report_date BETWEEN '2026-03-01' AND '2026-03-31'`).

In [8]:
import duckdb
from google.colab import userdata

# Connect to DuckDB
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Base URI for warehouse dataset
WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"

# Direct partition target for March 2026 (enables fast partition pruning)
FACT_MARCH_PATH = f"{WAREHOUSE_URI}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT_PATH = f"{WAREHOUSE_URI}/dim_content.parquet"

print("UNIT OF ANALYSIS : 1 row = (report_date, client_hash_id, content_hash_id)")
print("TIME WINDOW : March 1, 2026 to March 31, 2026")
print(f"TARGET PATH : {FACT_MARCH_PATH}")

UNIT OF ANALYSIS : 1 row = (report_date, client_hash_id, content_hash_id)
TIME WINDOW : March 1, 2026 to March 31, 2026
TARGET PATH : hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 2. Fields: feature / label / context / excluded

---

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every candidate column is assigned to one of four functional buckets to maintain temporal hygiene:

1. **Context / Keys (Metadata & Partitioning):** report_date, client_hash_id, content_hash_id, content_type, main_intent. Used for entity resolution, joins, and group-aware cross-validation splits.

2. **Feature Inputs (Knowable at decision moment $t$):** `word_count`, `search_volume`, `competition`, `cpc`, historical 7-day rolling average clicks (`hist_7d_avg_clicks` over $[t-7, t-1]$), historical 7-day rolling engagement (`hist_7d_sum_engagement` over $[t-7, t-1]$).

3. **Label / Target (Observed Outcome):** ``target_future_clicks_30d`` (cumulative organic clicks over forward interval $[t, t+29]$).

4. **Excluded (Dropped & Why):**
    - Raw URLs & Query Strings are excluded due to extreme cardinality, extreme sparsity, and privacy/compliance requirements.

    - `gsc_data_available` / `ga4_data_available` when `FALSE` are filtered out to avoid treating uncollected tracking data as zero performance.

    - Internal Health/Optimization Scores are excluded because downstream automated product outputs create circular logic and label leakage when used as predictor inputs.

In [12]:
# Programmatic verification of schema buckets
contract_schema = {
    "context": ["report_date", "client_hash_id", "content_hash_id", "content_type", "main_intent"],
    "features": ["word_count", "search_volume", "competition", "cpc", "hist_7d_avg_clicks", "hist_7d_sum_engagement"],
    "label": ["target_future_clicks_30d"],
    "excluded": {
        "raw_query_strings": "Privacy compliance & high cardinality sparsity",
        "raw_urls": "Key normalization & identifier privacy",
        "unintegrated_rows": "Prevents treating missing API data as zero traffic",
        "internal_health_scores": "Prevents circular logic and downstream label leakage"
    }
}

print("--- DATA CONTRACT SCHEMA BUCKETS ---")
print(f"Context Keys ({len(contract_schema['context'])}) : {contract_schema['context']}")
print(f"Feature Inputs ({len(contract_schema['features'])}) : {contract_schema['features']}")
print(f"Target Label ({len(contract_schema['label'])}) : {contract_schema['label']}")
print(f"Excluded Categories ({len(contract_schema['excluded'])}) : Defined with explicit rationale")

--- DATA CONTRACT SCHEMA BUCKETS ---
Context Keys (5) : ['report_date', 'client_hash_id', 'content_hash_id', 'content_type', 'main_intent']
Feature Inputs (6) : ['word_count', 'search_volume', 'competition', 'cpc', 'hist_7d_avg_clicks', 'hist_7d_sum_engagement']
Target Label (1) : ['target_future_clicks_30d']
Excluded Categories (4) : Defined with explicit rationale


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Grain Uniqueness Verification
query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(report_date, '_', client_hash_id, '_', content_hash_id)) AS unique_grain_keys,
    COUNT(*) - COUNT(DISTINCT CONCAT(report_date, '_', client_hash_id, '_', content_hash_id)) AS duplicate_count
FROM read_parquet('{FACT_MARCH_PATH}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
df_grain = con.sql(query_grain).df()
print("--- GRAIN UNIQUENESS VERIFICATION ---")
print(df_grain.to_string(index=False))

# Boundary and Entity Volume Counts
query_counts = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_slice_rows,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet('{FACT_MARCH_PATH}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
df_counts = con.sql(query_counts).df()
print("\n--- WINDOW BOUNDARIES AND ENTITY COUNTS ---")
print(df_counts.to_string(index=False))

# Integration Availability Filtering (using explicit IS TRUE logic)
query_availability = f"""
SELECT
    COUNT(*) AS total_rows_in_window,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_active_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_active_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) AS dual_active_rows,
    ROUND(100.0 * COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) / COUNT(*), 2) AS valid_retention_pct
FROM read_parquet('{FACT_MARCH_PATH}')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
df_availability = con.sql(query_availability).df()
print("\n--- INTEGRATION AVAILABILITY (IS TRUE) ---")
print(df_availability.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- GRAIN UNIQUENESS VERIFICATION ---
 total_rows  unique_grain_keys  duplicate_count
    9841378            9841378                0

--- WINDOW BOUNDARIES AND ENTITY COUNTS ---
  min_date   max_date  total_slice_rows  distinct_clients  distinct_content_items
2026-03-01 2026-03-31           9841378                55                  331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- INTEGRATION AVAILABILITY (IS TRUE) ---
 total_rows_in_window  gsc_active_rows  ga4_active_rows  dual_active_rows  valid_retention_pct
              9841378          3611061           413966            364347                  3.7


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Observed Data Limitations:**
1. **Unbalanced Historical Onboarding Lags:** Clients activate search integrations (GSC/GA4) at arbitrary calendar dates. Historical traffic prior to `gsc_data_start` or `ga4_data_start` does not retroactively backfill. Filtering on `IS TRUE` prevents false zeros but reduces total observational density during initial onboarding periods.

2. **GSC Privacy Threshold Masking:** Google Search Console enforces privacy thresholds that replace low-volume search queries with anonymized totals. This prevents granular keyword-level attribution for long-tail pages.

**Leakage Prevention Experiment:**
To prove temporal cleanliness, we extract valid historical features (lookback $[t-7, t-1]$) alongside a deliberate synthetic leakage trap column (containing same-day metrics $[t]$). We fit a baseline model with and without the trap to quantify artificial performance inflation before dropping the trap.

In [11]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# Extract Feature Frame for Mid-Panel Slice with rolling lookbacks [t-7, t-1]
df_features = con.sql(f"""
    SELECT
        f.report_date,
        f.client_hash_id,
        f.content_hash_id,
        c.word_count,
        c.search_volume,
        c.competition,

        -- Valid historical rolling metrics strictly over [t-7, t-1]
        AVG(f.gsc_clicks) OVER (
            PARTITION BY f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_avg_clicks,

        SUM(f.ga4_total_engagement_sec) OVER (
            PARTITION BY f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_sum_engagement,

        -- Forward Target: Next 7-day cumulative clicks [t, t+6]
        SUM(f.gsc_clicks) OVER (
            PARTITION BY f.content_hash_id
            ORDER BY f.report_date
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS target_future_clicks_7d,

        -- DELIBERATE LEAKAGE TRAP: Incorporates same-day/future performance
        (f.gsc_clicks * 7) + f.ga4_sessions AS LEAKED_current_performance_trap

    FROM read_parquet('{FACT_MARCH_PATH}') f
    LEFT JOIN read_parquet('{DIM_CONTENT_PATH}') c
        ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND f.gsc_data_available IS TRUE
      AND f.ga4_data_available IS TRUE
""").df().fillna(0)

honest_features = ['word_count', 'search_volume', 'competition', 'hist_7d_avg_clicks', 'hist_7d_sum_engagement']
leaked_features = honest_features + ['LEAKED_current_performance_trap']
target_col = 'target_future_clicks_7d'

# Fit Model WITH Leakage Trap
model_leaked = Ridge().fit(df_features[leaked_features], df_features[target_col])
score_leaked = r2_score(df_features[target_col], model_leaked.predict(df_features[leaked_features]))

# Fit Honest Baseline Model WITHOUT Leakage Trap
model_honest = Ridge().fit(df_features[honest_features], df_features[target_col])
score_honest = r2_score(df_features[target_col], model_honest.predict(df_features[honest_features]))

print("--- LEAKAGE PREVENTION CHECK ---")
print(f"R² WITH Synthetic Target Leakage Trap : {score_leaked:.4f} (Artificially Inflated)")
print(f"R² WITHOUT Leakage Trap (Honest Model) : {score_honest:.4f} (Real Observational Signal)")

# Explicit Remediation: Drop the Leakage Trap
df_clean = df_features.drop(columns=['LEAKED_current_performance_trap'])
print(f"REMEDIATION: Leaked column dropped. Clean dataset contains {df_clean.shape[1]} valid columns.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- LEAKAGE PREVENTION CHECK ---
R² WITH Synthetic Target Leakage Trap : 0.7285 (Artificially Inflated)
R² WITHOUT Leakage Trap (Honest Model) : 0.6585 (Real Observational Signal)
REMEDIATION: Leaked column dropped. Clean dataset contains 9 valid columns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.